# Ground Truth — Single CSV + KML with Pole Info + Component Heights

**One run → `pole_inventory.csv` + `pole_inventory.kml`**

### Logic
- **LiDAR masks** → heights of CROSSARM, SECONDARY, TRANSFORMER
- **GSV masks** → heights of PRIMARY, NEUTRAL, DOWNGUY (+ cross-verifies others)
- LiDAR wins on conflict (3D source more precise)
- UTM (EPSG:32632) → WGS84 lat/lon

### Output columns in CSV
`pole_id · orig_id · lat · lon · X · Y · Z · pole_height_m ·
crossarm_h · secondary_h · transformer_h · primary_h · neutral_h · downguy_h`

### KML
Each placemark pin shows: `ID: X | H: Y.Ym`
Click any pin → balloon with full pole info + all component heights


In [ ]:
import os, cv2
import numpy as np
import pandas as pd
import laspy
from sklearn.decomposition import PCA
from scipy.spatial import cKDTree
from pyproj import Transformer

# ── Paths ─────────────────────────────────────────────────────────────────────
IN_LAS        = 'allwires_v1.las'
LIDAR_MASK_DIR = 'gt_lidar_masks'   # renamed LiDAR masks (from Cell 1 of prev notebook)
GSV_MASK_DIR   = 'gt_gsv_masks'     # renamed GSV masks
OUT_CSV        = 'pole_inventory.csv'

# ── Render geometry (must match labeller) ─────────────────────────────────────
FIXED_U_M = 80.0;  FIXED_Z_M = 22.0
IMG_W     = 800;   IMG_H     = 220
SCALE_MPX = FIXED_U_M / IMG_W   # 0.1 m/px

# ── Label IDs ─────────────────────────────────────────────────────────────────
LABEL = {1:'SHAFT',2:'CROSSARM',3:'PRIMARY',4:'SECONDARY',
         5:'NEUTRAL',6:'DOWNGUY',7:'ANCHOR',8:'TRANSFORMER'}

# Components each mask source is trusted for
LIDAR_TRUSTED = {2,4,8}          # CROSSARM, SECONDARY, TRANSFORMER
GSV_TRUSTED   = {2,3,4,5,6,8}   # PRIMARY, NEUTRAL, DOWNGUY + cross-verify others

# ── Mapping: consecutive new_id → original LAS uid ───────────────────────────
orig_ids = [1,2,4,5,6,7,10,11,12,13,14,15,16,17,18,20,23,24,25,26,28,29,
            30,31,32,33,34,37,38,39,41,42,43,44,45,46,47]
NEW_TO_ORIG = {new: orig for new, orig in enumerate(orig_ids, start=1)}

# ── Load LAS ──────────────────────────────────────────────────────────────────
print(f'Loading {IN_LAS}...')
las      = laspy.read(IN_LAS)
pts      = np.vstack([las.x, las.y, las.z]).T
cls_arr  = np.array(las.classification, dtype=np.uint8)
uid_arr  = np.array(las.user_data,      dtype=np.int32)
pole_mask = cls_arr == 65

# ── UTM → WGS84 converter ─────────────────────────────────────────────────────
utm2wgs = Transformer.from_crs("EPSG:32632", "EPSG:4326", always_xy=True)

# ── Global road PCA (stable direction for all poles) ─────────────────────────
all_pole_xy = np.array([
    pts[pole_mask & (uid_arr == oid), :2].mean(axis=0)
    for oid in orig_ids
])
road_pca   = PCA(n_components=2).fit(all_pole_xy)
road_u_vec = road_pca.components_[0]

# ── Full 2D KDTree ────────────────────────────────────────────────────────────
full_tree = cKDTree(pts[:, :2])

print(f'Setup done. Processing {len(orig_ids)} poles...')
print()

# ── Helper: get component heights from a mask ─────────────────────────────────
def get_heights_from_mask(mask_path, pole_xy, bot_z, z_lo,
                          near_gidx, u_phys, z_phys, trusted_labels):
    """
    Returns dict: {label_id: height_above_ground_m} for trusted labels only.
    """
    if not os.path.exists(mask_path):
        return {}
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    if mask is None or mask.max() == 0:
        return {}

    u_ctr = IMG_W // 2
    ux = np.clip((u_ctr + u_phys / SCALE_MPX).astype(int), 0, IMG_W - 1)
    vx = np.clip(((IMG_H-1) - (z_phys - z_lo) / SCALE_MPX).astype(int), 0, IMG_H - 1)
    painted = mask[vx, ux]

    heights = {}
    for label_id in trusted_labels:
        m = painted == label_id
        if not m.any():
            continue
        median_z = float(np.median(z_phys[m]))
        heights[label_id] = round(median_z - bot_z, 2)
    return heights

# ── Main loop ─────────────────────────────────────────────────────────────────
rows = []

for new_id, orig_pid in NEW_TO_ORIG.items():
    m_pole = pole_mask & (uid_arr == orig_pid)
    if not m_pole.any():
        print(f'  Pole {new_id:02d}: no POLE points in LAS — skipped')
        continue

    p_pole  = pts[m_pole]
    pole_xy = p_pole[:, :2].mean(axis=0)
    bot_z   = float(p_pole[:, 2].min())    # ground reference
    top_z   = float(p_pole[:, 2].max())
    mid_z   = float(p_pole[:, 2].mean())
    pole_h  = round(top_z - bot_z, 2)

    # Lat/lon
    lon, lat = utm2wgs.transform(pole_xy[0], pole_xy[1])

    # Pull nearby points for projection
    z_lo    = bot_z - 1.0
    raw_local = np.array(full_tree.query_ball_point(pole_xy, r=FIXED_U_M/2+3), dtype=np.int64)
    near_pts  = pts[raw_local]
    zm = (near_pts[:,2] >= z_lo) & (near_pts[:,2] <= z_lo + FIXED_Z_M)
    near_pts  = near_pts[zm]
    near_gidx = raw_local[zm]
    if len(near_pts) == 0:
        continue

    rel_xy  = near_pts[:, :2] - pole_xy
    u_phys  = rel_xy @ road_u_vec
    z_phys  = near_pts[:, 2]

    # ── LiDAR mask: CROSSARM, SECONDARY, TRANSFORMER ──────────────────────────
    lidar_path = os.path.join(LIDAR_MASK_DIR, f'pole_{new_id:03d}_mask.png')
    lidar_h    = get_heights_from_mask(
        lidar_path, pole_xy, bot_z, z_lo,
        near_gidx, u_phys, z_phys, LIDAR_TRUSTED
    )

    # ── GSV mask: PRIMARY, NEUTRAL, DOWNGUY + cross-verify others ─────────────
    gsv_path = os.path.join(GSV_MASK_DIR, f'pole_{new_id:03d}_mask.png')
    gsv_h    = get_heights_from_mask(
        gsv_path, pole_xy, bot_z, z_lo,
        near_gidx, u_phys, z_phys, GSV_TRUSTED
    )

    # ── Merge: LiDAR wins for shared components ────────────────────────────────
    merged = {}
    merged.update(gsv_h)    # GSV base
    merged.update(lidar_h)  # LiDAR overwrites where it has data

    # ── Build row ─────────────────────────────────────────────────────────────
    row = {
        'pole_id':          new_id,
        'orig_id':          orig_pid,
        'lat':              round(lat, 7),
        'lon':              round(lon, 7),
        'X':                round(pole_xy[0], 3),
        'Y':                round(pole_xy[1], 3),
        'Z':                round(mid_z, 3),
        'pole_height_m':    pole_h,
        # Component heights above ground (None if not labelled)
        'crossarm_h_m':     merged.get(2, None),
        'secondary_h_m':    merged.get(4, None),
        'transformer_h_m':  merged.get(8, None),
        'primary_h_m':      merged.get(3, None),
        'neutral_h_m':      merged.get(5, None),
        'downguy_h_m':      merged.get(6, None),
    }
    rows.append(row)

    # Print summary
    comp_str = '  '.join(
        f'{name}={merged[lid]:.1f}m'
        for lid, name in [(2,'CA'),(4,'SEC'),(8,'XFMR'),(3,'PRI'),(5,'NEU'),(6,'GUY')]
        if lid in merged
    )
    print(f'  Pole {new_id:02d}: H={pole_h:.1f}m  lat={lat:.5f} lon={lon:.5f}  [{comp_str}]')

# ── Save single CSV ───────────────────────────────────────────────────────────
df = pd.DataFrame(rows)
df.to_csv(OUT_CSV, index=False)

print(f'\n✅  {OUT_CSV} saved — {len(df)} poles')
print()
print(df.to_string(index=False))

# ── Generate KML ──────────────────────────────────────────────────────────────
OUT_KML = 'pole_inventory.kml'

def component_rows_html(row):
    """Build a neat HTML table of component heights for KML balloon."""
    comps = [
        ('Crossarm',    row.get('crossarm_h_m')),
        ('Secondary',   row.get('secondary_h_m')),
        ('Transformer', row.get('transformer_h_m')),
        ('Primary',     row.get('primary_h_m')),
        ('Neutral',     row.get('neutral_h_m')),
        ('Downguy',     row.get('downguy_h_m')),
    ]
    rows_html = ''
    for name, val in comps:
        if val is None or (isinstance(val, float) and pd.isna(val)):
            continue
        rows_html += (
            f'<tr>'
            f'<td style="padding:2px 8px;color:#555">{name}</td>'
            f'<td style="padding:2px 8px;font-weight:bold">{val:.2f} m</td>'
            f'</tr>'
        )
    return rows_html

kml_placemarks = []
for _, row in df.iterrows():
    pid        = int(row['pole_id'])
    orig_id    = int(row['orig_id'])
    lat        = float(row['lat'])
    lon        = float(row['lon'])
    altitude   = float(row['Z'])
    pole_h     = float(row['pole_height_m'])

    comp_html = component_rows_html(row)

    description = f"""<![CDATA[
    <b style="font-size:14px">Pole {pid}</b>
    <table style="border-collapse:collapse;margin-top:6px;font-family:Arial;font-size:12px">
      <tr><td style="padding:2px 8px;color:#555">Pole ID</td>
          <td style="padding:2px 8px;font-weight:bold">{pid}</td></tr>
      <tr><td style="padding:2px 8px;color:#555">Orig ID</td>
          <td style="padding:2px 8px">{orig_id}</td></tr>
      <tr><td style="padding:2px 8px;color:#555">Height</td>
          <td style="padding:2px 8px;font-weight:bold">{pole_h:.1f} m</td></tr>
      <tr><td style="padding:2px 8px;color:#555">Lat</td>
          <td style="padding:2px 8px">{lat:.7f}</td></tr>
      <tr><td style="padding:2px 8px;color:#555">Lon</td>
          <td style="padding:2px 8px">{lon:.7f}</td></tr>
      <tr><td colspan="2" style="padding:4px 8px;color:#333;font-weight:bold;
          border-top:1px solid #ccc">Component Heights above Ground</td></tr>
      {comp_html}
    </table>
    ]]>"""

    placemark = f"""  <Placemark>
    <name>ID: {pid} | H: {pole_h:.1f}m</name>
    <description>{description}</description>
    <styleUrl>#poleStyle</styleUrl>
    <Point>
      <altitudeMode>absolute</altitudeMode>
      <coordinates>{lon:.7f},{lat:.7f},{altitude:.3f}</coordinates>
    </Point>
  </Placemark>"""
    kml_placemarks.append(placemark)

kml_body = '\n'.join(kml_placemarks)

kml_content = f"""<?xml version="1.0" encoding="UTF-8"?>
<kml xmlns="http://www.opengis.net/kml/2.2">
<Document>
  <name>Pole Inventory</name>
  <description>Utility poles with component heights from LiDAR</description>

  <!-- Pole marker style -->
  <Style id="poleStyle">
    <IconStyle>
      <color>ff1478ff</color>
      <scale>0.9</scale>
      <Icon>
        <href>http://maps.google.com/mapfiles/kml/shapes/placemark_circle.png</href>
      </Icon>
    </IconStyle>
    <LabelStyle>
      <color>ff1478ff</color>
      <scale>0.75</scale>
    </LabelStyle>
    <BalloonStyle>
      <text>$[description]</text>
      <bgColor>ffffffff</bgColor>
    </BalloonStyle>
  </Style>

{kml_body}
</Document>
</kml>"""

with open(OUT_KML, 'w', encoding='utf-8') as f:
    f.write(kml_content)

print(f'✅  {OUT_KML} saved — {len(df)} placemarks')
print(f'    Open in Google Earth → each pin shows ID, height, and all component heights')

